In [4]:
import os
import pandas as pd
import toml

config = toml.load('config.toml')

# Load a parcel to tract lookup for aggregating Daysim records to tracts
# Load the sqlite lite table from the model
import os
import sqlite3
con = sqlite3.connect(
            os.path.join(
                config["model_run_dir_2023"],
                r"inputs/db/soundcast_inputs_2023.db",
            )
        )
# conn = os.path.join(config["model_run_dir_2023"], "inputs\db\soundcast_inputs_2023.db")

df_db = pd.read_sql(con=con,
                    sql="SELECT ParcelID, GEOID20, Census2020Block FROM parcel_2023_geography")

In [ ]:
df_db[["GEOID20","Census2020Block"]]

,GEOID20,Census2020Block
0,5.303303e+14,5.303303e+14
1,5.303303e+14,5.303303e+14
2,5.303303e+14,5.303303e+14
3,5.303303e+14,5.303303e+14
4,5.303303e+14,5.303303e+14
...,...,...
1329923,5.306105e+14,5.306105e+14
1329924,5.306105e+14,5.306105e+14
1329925,5.306105e+14,5.306105e+14
1329926,5.306105e+14,5.306105e+14


In [6]:
df_db["GEOID20"].nunique()

47206

In [ ]:
df_trip = pd.read_csv(os.path.join(config["model_run_dir_2023"], "outputs/daysim/_trip.tsv"), sep="\t")

# Filter for trips not ending at home
df_nonhome_trip = df_trip[df_trip["dpurp"] != 0]

# select only vehicle trips
df_nonhome_trip = df_nonhome_trip[df_nonhome_trip["mode"].isin([3,4,5]) & (df_nonhome_trip["dorp"] == 1)]

# Merge tract geoid to dpcl
df_nonhome_trip = df_nonhome_trip.merge(df_db, left_on="dpcl", right_on="ParcelID", how="left")

nonhome_trips_by_tract = df_nonhome_trip.groupby("GEOID20").size().reset_index(name="tract_nonhome_trips_in_vehicle")
nonhome_trips_by_tract["geoid20"] = nonhome_trips_by_tract["GEOID20"].astype("int64").astype("str")

nonhome_trips_by_tract.to_csv(os.path.join(config["working_dir"], "nonhome_trips_dest_by_tract.csv"), index=False)